# Objective: Build a machine learning pipeline to detect fraudulent financial transactions from a heavily imbalanced dataset, addressing class imbalance as a core challenge. 

### Tech Stack: Python, pandas, scikit-learn, imbalanced-learn (SMOTE), matplotlib, seaborn, Jupyter Notebook 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,confusion_matrix,ConfusionMatrixDisplay,precision_score,recall_score,f1_score,roc_auc_score,
    RocCurveDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df=pd.read_csv('data\credit_card_fraud_10k.csv')

In [ ]:
df.head()

In [ ]:
#rows and columns
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#checking for null values
df.isnull().sum()

## Understanding Class Imbalance

Fraud detection datasets are usually highly imbalanced because fraudulent
transactions are very rare compared to legitimate transactions.

This imbalance makes the learning task difficult because machine learning
models tend to favor the majority class (non-fraud), potentially ignoring
fraudulent cases.

Therefore, special evaluation metrics and balancing techniques are required
instead of relying solely on accuracy.

In [ ]:
#checking class distribution
df["is_fraud"].value_counts()

In [ ]:
fraud_percentage = df["is_fraud"].mean() * 100
print(f"Fraudulent Transactions : {fraud_percentage:.2f}%")

In [ ]:
plt.figure(figsize=(5,4))

sns.countplot(data=df, x="is_fraud")

plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Fraud")
plt.ylabel("Count")
plt.savefig('images/fraud_vs_nonfraud.png')
plt.show()

The dataset is highly imbalanced, with only a small percentage of transactions
belonging to the fraud class.
This imbalance requires special handling because most machine learning models
naturally become biased toward the majority class.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='is_fraud', y='amount')
plt.title('Transaction amount distribution')
plt.savefig('images/transaction_amt_distr.png')

### Time day analysis

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(data= df, x='transaction_hour', hue='is_fraud')
plt.title("fraud by transaction hour")
plt.savefig('images/distribution_fraud_by_hour.png')
plt.show()

The distribution of legitimate transactions remains relatively consistent across all 24 hours, indicating that normal customer activity occurs throughout the day.

Fraudulent transactions are much less frequent due to the highly imbalanced nature of the dataset (1.51% fraud rate). A slightly higher concentration of fraudulent transactions can be observed during the early morning hours (approximately 12 AM to 3 AM), while fraud cases remain relatively low during the rest of the day.

Although transaction hour alone is not a strong indicator of fraud, it may provide useful contextual information when combined with other features in the machine learning model.

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(
    data=df[df["is_fraud"] == 1],
    x="transaction_hour",
    color="crimson"
)

plt.title("Distribution of Fraudulent Transactions by Hour")
plt.xlabel("Transaction Hour")
plt.ylabel("Number of Fraud Transactions")
plt.savefig('images/only_fraud_trans_hour.png')
plt.show()

## Why Accuracy is Misleading

In fraud detection, fraudulent transactions usually represent only a very small
portion of the dataset.

A model that predicts every transaction as legitimate may achieve extremely
high accuracy while completely failing to detect fraud.

For this reason, metrics such as Precision, Recall, F1-Score, and ROC-AUC are
more informative than accuracy.

In [ ]:
X= df.drop(['transaction_id','is_fraud'], axis=1)
y= df['is_fraud']

In [ ]:
categorical_cols= ['merchant_category']
numerical_cols= X.select_dtypes(include=np.number).columns

In [ ]:
preprocessor= ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown='ignore'),categorical_cols),
        ("num", SimpleImputer(strategy='median'), numerical_cols)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(X,y, random_state=42, test_size=0.2, stratify=y)

In [ ]:
logistic_pipeline= ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
rf_pipeline= ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model', RandomForestClassifier(random_state=42))
])

In [ ]:
#training
logistic_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

In [ ]:
lg_pred=logistic_pipeline.predict(X_test)
rf_pred= rf_pipeline.predict(X_test)

## Recall vs Precision in Fraud Detection

In fraud detection, **Recall is generally considered the most important metric** because failing to detect a fraudulent transaction can result in direct financial losses and security risks.

A higher Recall ensures that most fraud cases are identified, even if this increases the number of false alarms. However, excessively low Precision can inconvenience genuine customers by incorrectly flagging legitimate transactions.

Therefore, the ideal fraud detection system seeks a balance between Recall and Precision. Metrics such as the F1-Score and ROC-AUC help evaluate this trade-off more effectively than accuracy alone.

In [ ]:
print(classification_report(y_test, lg_pred))

In [ ]:
print(classification_report(y_test, rf_pred))

Random Forest achieved significantly higher Precision and F1-Score, meaning that most fraud predictions were correct. However, it detected fewer fraudulent transactions than Logistic Regression, leading to a lower Recall.

Overall, Random Forest produced a better balance between identifying fraud and minimizing false alarms, while Logistic Regression prioritized detecting nearly all fraud cases at the cost of more false positives.

In [ ]:
print("Precision :", precision_score(y_test, rf_pred))
print("Recall :", recall_score(y_test, rf_pred))
print("F1 :", f1_score(y_test, rf_pred))
print("ROC AUC :", roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:,1]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    cmap='Blues'
)
plt.savefig('images/rf_confusion_matrix.png')
plt.show()

ROC Curve

In [ ]:
RocCurveDisplay.from_estimator(
    rf_pipeline,
    X_test,
    y_test
)

plt.show()

The ROC curve lies very close to the upper-left corner of the graph, indicating excellent classification performance. This means the model achieves a high True Positive Rate (correctly detecting fraudulent transactions) while maintaining a very low False Positive Rate (incorrectly flagging legitimate transactions).

The model achieved an **AUC score of 1.00**, suggesting an almost perfect ability to distinguish between fraudulent and legitimate transactions on this dataset. Such a high score indicates that the selected features provide strong discriminatory power for fraud detection.

Although an AUC of 1.00 is excellent, it is relatively uncommon in real-world fraud detection problems and may be influenced by the characteristics of this dataset.

### Feauture Extraction

In [ ]:
feature_names= logistic_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefficients= logistic_pipeline.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df = coef_df.sort_values(by="Coefficient", key=abs, ascending=False)

coef_df.head(10)

In [ ]:
importance = rf_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(20,6))

sns.barplot(
    data=importance_df.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top 10 Feature Importances")
plt.savefig('images/imp_features.png')
plt.show()

Feature importance analysis helps identify which variables contribute most to
fraud detection.

Features with higher importance have a stronger influence on the model's
predictions and can provide useful business insights for fraud prevention.

## Scalability

If deployed in a real banking environment handling one million transactions per
hour, the model would require a scalable inference pipeline.

Possible improvements include:

- Deploying the trained model as a REST API or microservice.
- Using distributed stream processing platforms such as Apache Kafka and Apache Spark.
- Performing predictions in parallel across multiple servers.
- Continuously monitoring model performance and retraining with newly observed fraud patterns.
- Applying real-time feature engineering and model caching to reduce prediction latency.

These techniques enable high-throughput, low-latency fraud detection suitable
for production-scale financial systems.

# Conclusion

This project successfully developed a machine learning pipeline to detect fraudulent financial transactions while addressing the challenge of class imbalance.

The dataset contained only **1.51% fraudulent transactions**, making it a highly imbalanced classification problem. To improve the model's ability to learn minority-class patterns, **SMOTE (Synthetic Minority Oversampling Technique)** was applied to the training data.

Two classification algorithms were evaluated:

- **Logistic Regression**
- **Random Forest Classifier**

Logistic Regression achieved a very high **Recall (93%)**, successfully identifying most fraudulent transactions but producing a larger number of false positives. In contrast, Random Forest achieved significantly higher **Precision (94%)** and a stronger **F1-Score (71%)**, indicating a better balance between fraud detection and minimizing false alarms.

The project also demonstrated why **accuracy alone is not an appropriate evaluation metric** for fraud detection. Instead, Precision, Recall, F1-Score, and ROC-AUC provided a much more meaningful assessment of model performance.

Feature importance analysis highlighted the variables that contributed most to fraud prediction, offering valuable business insights into transaction risk. Finally, scalability considerations showed how such a solution could be extended to real-time financial systems using distributed processing frameworks such as Apache Kafka and Apache Spark.

Overall, this project provided practical experience in handling imbalanced datasets, applying data preprocessing techniques, evaluating classification models using appropriate metrics, and understanding the business implications of fraud detection systems.